# Cell Insertion, Desktop Quick-Pickers, Slash-Morph & Image Drop (`CRAFT.js`, `CRAFT.css`)

> Every way a desktop user creates or morphs cells: the active cell-type picker (`window.craftInsertPreview`), hover-edge drag-to-insert and 4-option fan picker, `Shift+A`/`Shift+B` seam quick-picker, `/c` `/n` `/p` `/r` slash-morph and click-margin escape (`window.craftDesktopUX`), and native Monaco image-drop handoff (`window.craftImageDrop`).

## Insertion Templates & Cell-Type Config (`#| export_ft`)

Single source of truth for cell-type SVG icons, insertion triggers, fan picker markup, and the `key_to_type` / `slash_map` tables exposed in `window.__craftCfg`.

In [ ]:
#| export_ft CRAFT.js
#| eval: false
CELL_PATHS = dict(
    Code='M8 6l-6 6 6 6m8-12 6 6-6 6M14 4l-4 16',
    Note='M5 3h10l4 4v14H5z M15 3v5h4M8 12h8m-8 4h8',
    Raw='M3 5h18v14H3z M7 9v6',
    Prompt='M3 4h18v12H9l-6 4z M7 10h.01m5 0h.01m5 0h.01',
)
DESK_ORDER = ('Code', 'Note', 'Prompt', 'Raw')

TEMPLATES = merge(
    dict(
        insert_trigger=Button(id='craft-insert-preview', klass='craft-nav-icon-btn', type='button', hidden=True, popovertarget='craft-insert-preview-menu'),
        insert_menu=craft_menu(
            Small('New cell type'),
            *[Button(svg_icon(p, 15, 1.6), name, type='button', data_kind=name) for name, p in CELL_PATHS.items()],
            id='craft-insert-preview-menu'),
        desk_edge_below=Button(
            Span(svg_icon('M12 6v12M6 12h12', 12, 1.8), Span(klass='craft-desk-edge-kind'), klass='craft-desk-edge-pill'),
            type='button', klass='craft-desk-edge', data_edge='below',
            aria_label='Insert cell below (drag to pick type)'),
        desk_picker=Div(
            Div(klass='craft-type-picker-head craft-desk-picker-head'),
            Div(*[craft_btn(svg_icon(CELL_PATHS[t], 14, 1.8), Span(t), ft('kbd', t[0]),
                            role='menuitem', data_type=t) for t in DESK_ORDER],
                klass='craft-type-picker-opts craft-desk-picker-opts'),
            id='craft-desk-picker', klass='craft-type-picker', popover='auto', role='menu'),
        optimistic_cell=Div(
            Div(
                Div(Div(klass='msg-content'), klass='msg-body'),
                Form(Textarea(name='content'), hidden=True),
                klass='uk-card uk-card-default msg-card card-in',
            ),
            klass='editable craft-opt-cell',
            data_craft_optimistic='1',
        ),
        empty_add=Div(
            Span('Empty dialog — press a/b or add your first cell', klass='craft-empty-add-label'),
            Div(*[craft_btn(svg_icon(CELL_PATHS[t], 15, 1.7), Span(t), data_type=t) for t in DESK_ORDER],
                klass='craft-empty-add-btns'),
            id='craft-empty-add'),
    ),
    {f'cell_icon_{k}': svg_icon(p, 15, 1.6) for k, p in CELL_PATHS.items()},
    {f'desk_icon_{k}': svg_icon(p, 11, 1.8) for k, p in CELL_PATHS.items()},
)

CONFIG = dict(
    cell_types=list(DESK_ORDER),
    key_to_type={t[0].lower(): t for t in DESK_ORDER},
    slash_map={k: t for t in DESK_ORDER for k in (t[0].lower(), t.lower())},
)

## Insert Type Picker (`window.craftInsertPreview`)

Controls which cell type `a` and `b` insert, renders the shimmer placeholder while waiting for Solveit's WebSocket card refresh, and hands focus to `craftC2` or `craftMd` once the new card arrives.

In [ ]:
#| export_js CRAFT.js
#| eval: false
(() => {
  window.craftInsertPreview?.disable();
  if (!window.__craftCMCore.features.init('insert-preview', 'Insert type picker', 'Typed A/B optimistic cell insertion')) return;
  if (!window.__craftCMCore.qs(window.__craftCMCore.SOLVEIT.sel.dialog)) return;
  const types = {Code: 1, Note: 1, Raw: 1, Prompt: 1};
  const icon = name => window.__craftTpl?.[`cell_icon_${name}`] || '';
  const key = 'craft-insert-preview:v1:' + (new URLSearchParams(location.search).get('name') || location.pathname);
  const {mountTpl, qs, qsa, on, store, patch, onDomChange, SOLVEIT, allCells} = window.__craftCMCore;
  const ac = new AbortController(), sig = {signal: ac.signal}, cap = {capture: true, signal: ac.signal};
  const trigger = mountTpl('insert_trigger'), menu = mountTpl('insert_menu'), emptyAdd = mountTpl('empty_add');
  let current;
  const select = name => {
    current = name;
    const title = `A/B inserts: ${name}`;
    patch(trigger, {innerHTML: icon(name), title, 'aria-label': title});
    for (const option of qsa('button', menu)) option.setAttribute('aria-pressed', String(option.dataset.kind === name));
  };
  const saved = store.get(key);
  select(Object.hasOwn(types, saved) ? saved : document.getElementById('default_code')?.checked ? 'Code' : 'Prompt');
  let scroller;
  const sync = () => {
    const next = qs(SOLVEIT.sel.dialog);
    if (scroller !== next) {
      scroller?.classList.remove('craft-insert-preview');
      scroller = next;
      trigger.hidden = !scroller;
      scroller?.classList.add('craft-insert-preview');
      window.craftMobile?.reposition?.();
    }
    if (scroller && !emptyAdd.isConnected) scroller.append(emptyAdd);
    else if (!scroller && emptyAdd.isConnected) emptyAdd.remove();
  };
  const choose = name => {
    select(name);
    store.set(key, name);
  };
  const activeCells = () => allCells().filter(el => el.matches(SOLVEIT.sel.selectedCell) && el.getClientRects().length);
  const cellType = el => el?.querySelector(SOLVEIT.sel.cardInType)?.dataset.mtype;
  const notify = error => {
    console.error('CRAFT insertion:', error);
    window.__craftCMCore.notify(error.message, {status: 'danger'});
  };
  const waitFor = (check, timeout = 5000) => {
    if (ac.signal.aborted) return Promise.resolve(null);
    const immediate = check();
    if (immediate) return Promise.resolve(immediate);
    return new Promise(resolve => {
      let done = false;
      const onAbort = () => finish(null);
      const finish = val => {
        if (done) return;
        done = true;
        ac.signal.removeEventListener('abort', onAbort);
        offDomWait();
        clearInterval(tick);
        clearTimeout(cap);
        resolve(val);
      };
      const probe = () => { const r = check(); if (r) finish(r); };
      ac.signal.addEventListener('abort', onAbort, {once: true});
      const offDomWait = onDomChange(probe, {attrs: ['data-mtype', 'class', 'hidden', 'data-editing'], delay: 0});
      const tick = setInterval(probe, 100);
      const cap = setTimeout(() => finish(null), timeout);
    });
  };
  const setType = async (id, type) => {
    if (cellType(document.getElementById(id)) !== type) {
      await window.__craftCMCore.htmxPost(SOLVEIT.api.setMtyp, {id_: id, ids: id, mtyp: type});
      if (!(await waitFor(() => cellType(document.getElementById(id)) === type))) throw new Error(`Could not switch cell to ${type}.`);
    }
  };
  const blocked = () => window.__craftCMCore.typingBlocked('.cm-editor,#craft-insert-preview-menu');
  let busy = false;
  const execute = action => {
    if (!busy) {
      busy = true;
      Promise.resolve().then(action).catch(notify).finally(() => { busy = false; });
    }
    return true;
  };
  const mode = name => {
    if (blocked()) return false;
    choose(name);
    const cells = activeCells();
    if (cells.length !== 1) return true;
    return execute(async () => {
      let id = cells[0].id;
      if (window.__craftCMCore.optimistic?.isOptimistic?.(id)) {
        const realId = await window.__craftCMCore.optimistic.waitForReal(id);
        if (!realId) return;
        id = '_' + realId;
      }
      await setType(id, name.toLowerCase());
    });
  };
  let optSeq = 0;
  let serverQueue = Promise.resolve();
  const isRealCell = el => Boolean(el?.id && !el.dataset.craftOptimistic);
  const realCells = () => allCells().filter(isRealCell);

  // opts.type ('Code'|'Note'|'Prompt'|'Raw') overrides the picker; opts.cell targets a cell other than the
  // selection; opts.force skips the focus guard (the mobile toolbar has already closed any editor).
  const insert = (where, opts = {}) => {
    if (!opts.force && blocked()) return false;
    const primary = opts.cell || activeCells().find(el => el.dataset.sm === 'primary');
    const isFirst = !primary && allCells().length === 0;
    if (!primary && !isFirst) return true;
    const addSel = `[hx-post="${where === 'above' ? SOLVEIT.api.addAbove : SOLVEIT.api.addBelow}"]`;
    if (primary && !primary.dataset.craftOptimistic && !primary.querySelector(addSel)) return true;
    const container = qs(SOLVEIT.sel.dialog);
    if (isFirst && !container) return true;
    const name = Object.hasOwn(types, opts.type || '') ? opts.type : current;
    const type = name.toLowerCase();
    const CORE = window.__craftCMCore;
    const optId = 'craftopt_' + (++optSeq);
    const optEl = mountTpl('optimistic_cell');
    optEl.id = '_' + optId;
    const optCard = optEl.querySelector(SOLVEIT.sel.cardIn);
    if (optCard) optCard.dataset.mtype = type;
    if (isFirst) container.append(optEl); else if (where === 'above') primary.before(optEl); else primary.after(optEl);
    window.selectMsg?.(window.$(optEl), {scroll: false});
    optEl.scrollIntoView({block: 'nearest'});

    let resolveOpt;
    const optPromise = new Promise(r => { resolveOpt = r; });
    CORE.optimistic.register(optId, optPromise);

    const openP = type === 'code'
      ? Promise.resolve(window.craftC2?.edit?.(optId, {anchor: 0, head: 0}))
      : Promise.resolve(window.craftMd?.edit?.(optId, {toEnd: true}));

    const runSync = async () => {
      const releaseAutoEdit = CORE.optimistic.suppressAutoEdit();
      let oldReal = new Set();
      let realDomId = null;
      const hidden = new Set();
      const hideIncoming = () => {
        for (const c of realCells()) {
          if (!oldReal.has(c.id)) {
            if (!c.classList.contains('craft-opt-hidden')) c.classList.add('craft-opt-hidden');
            hidden.add(c);
          }
        }
      };
      let offHide = null;
      try {
        if (isFirst) {
          oldReal = new Set(realCells().map(el => el.id));
          offHide = onDomChange(hideIncoming, {delay: 0, inDialog: true});
          await CORE.htmxPost(SOLVEIT.api.upsertMsg, {id_: '', msg_type: type, is_input: '1', content: ' '});
        } else {
          let anchorEl = primary;
          if (primary.dataset.craftOptimistic) {
            const anchorRealId = await CORE.optimistic.waitForReal(primary.id);
            if (!anchorRealId) throw new Error(`Could not confirm ${name} insertion.`);
            anchorEl = document.getElementById('_' + anchorRealId);
          }
          const button = anchorEl?.querySelector(addSel);
          if (!button) throw new Error(`Could not confirm ${name} insertion.`);
          oldReal = new Set(realCells().map(el => el.id));
          offHide = onDomChange(hideIncoming, {delay: 0, inDialog: true});
          button.dispatchEvent(new MouseEvent('click', {bubbles: false}));
        }
        realDomId = await waitFor(() => {
          hideIncoming();
          return realCells().find(el => !oldReal.has(el.id))?.id;
        });
        if (!realDomId) throw new Error(`Could not confirm ${name} insertion.`);
        hideIncoming();
        if (cellType(document.getElementById(realDomId)) !== type) await setType(realDomId, type);
        const realEl = document.getElementById(realDomId);
        if (!realEl) throw new Error(`Could not confirm ${name} insertion.`);
        if (realEl.dataset.editing) delete realEl.dataset.editing;
        await openP;
        const realNormId = CORE.normId(realDomId);
        const realCard = realEl.querySelector(SOLVEIT.sel.cardIn);
        const editorApi = type === 'code' ? window.craftC2 : window.craftMd;
        const optKey = type === 'code' ? optId : `${optId}:in`;
        const realKey = type === 'code' ? realNormId : `${realNormId}:in`;
        const hadView = Boolean(editorApi?.view?.(optKey));
        realEl.classList.remove('craft-opt-hidden');
        hidden.delete(realEl);
        if (hadView) {
          editorApi.rekey(optId, realNormId, {el: realEl, card: realCard});
        } else {
          window.selectMsg?.(window.$(realEl), {scroll: false});
        }
        optEl.remove();
        CORE.optimistic.resolve(optId, realNormId);
        resolveOpt(realNormId);
        if (hadView && editorApi.active?.() === realKey) {
          window.selectMsg?.(window.$(realEl), {scroll: false});
        }
      } catch (err) {
        for (const c of hidden) c.classList.remove('craft-opt-hidden');
        const editorApi = type === 'code' ? window.craftC2 : window.craftMd;
        const optKey = type === 'code' ? optId : `${optId}:in`;
        const v = editorApi?.view?.(optKey);
        const draft = v ? v.state.doc.toString() : '';
        CORE.optimistic.fail(optId);
        resolveOpt(null);
        editorApi?.abortOptimistic?.(optId);
        optEl.remove();
        CORE.optimistic.clear(optId);
        if (primary?.isConnected) window.selectMsg?.(window.$(primary), {scroll: false});
        let copied = false;
        if (draft.trim() && navigator.clipboard?.writeText) {
          try { await navigator.clipboard.writeText(draft); copied = true; } catch {}
        }
        const msg = copied ? `${err.message} Your draft was copied to the clipboard.` : err.message;
        notify(new Error(msg));
      } finally {
        offHide?.();
        for (const c of hidden) c.classList.remove('craft-opt-hidden');
        releaseAutoEdit();
      }
    };
    serverQueue = serverQueue.then(runSync, runSync);
    return true;
  };
  const defs = [
    ['si-a', 'a', 'Add message above', () => insert('above')],
    ['si-b', 'b', 'Add message below', () => insert('below')],
    ...['Code','Note','Prompt','Raw'].map((name,i) => [`si-mode-${name.toLowerCase()}`, `Mod+Shift+${'JKL;'[i]}`, `Switch message to ${name.toLowerCase()}`, () => mode(name)]),
  ].map(([id,combo,label,handler]) => ({id,combo,label,handler,group:'Solveit · Dialog (selection mode)'}));
  const registerBindings = () => defs.forEach(spec => window.__hotkeyRegister?.(spec));
  const builtins = qs('script[data-solveit-hotkeys="builtins"]');
  on(builtins, 'load', registerBindings, {once: true, signal: ac.signal});
  registerBindings();
  on(menu, 'click', e => {
    const option = e.target.closest('button[data-kind]');
    if (!option) return;
    choose(option.dataset.kind);
    menu.hidePopover(); trigger.focus();
  }, sig);
  document.body.append(trigger, menu);
  on(emptyAdd, 'click', e => {
    const opt = e.target.closest('button[data-type]');
    if (!opt) return;
    e.preventDefault(); e.stopPropagation();
    insert('below', {type: opt.dataset.type, force: true});
  }, sig);
  const KIND_MAP = {code: 'Code', note: 'Note', prompt: 'Prompt', raw: 'Raw'};
  const onHtmxConfig = e => {
    const path = e.detail?.path || '';
    const mtyp = e.detail?.parameters?.mtyp || e.detail?.parameters?.msg_type;
    if ((path.startsWith(SOLVEIT.api.setMtyp) || path.startsWith(SOLVEIT.api.upsertMsg)) && mtyp && KIND_MAP[String(mtyp).toLowerCase()]) {
      choose(KIND_MAP[String(mtyp).toLowerCase()]);
    }
  };
  on(document, 'htmx:configRequest', onHtmxConfig, sig);
  const offDom = onDomChange(sync, {delay: 0});
  window.craftInsertPreview = {
    insert: (where, opts) => insert(where, opts),
    choose,
    setType,
    type: () => current,
    isBusy: () => busy,
    disable() {
    ac.abort();
    offDom();
    defs.forEach(spec => window.__hotkeyRegister?.({...spec, builtin:true, handler:() => true}));
    scroller?.classList.remove('craft-insert-preview');
    trigger.remove(); menu.remove(); emptyAdd.remove(); delete window.craftInsertPreview;
  }};
  sync();
})();

## Desktop UX (`window.craftDesktopUX`)

Hover-edge `+` handle with drag-to-pick fan menu, `Shift+A`/`Shift+B` seam quick-picker, click-margin/header escape out of active inline editors, and `/c` `/n` `/p` `/r` slash-morph on empty cells.

In [ ]:
#| export_js CRAFT.js
#| eval: false
/* ── Desktop UX: hover-edge drag-to-insert, click-margin escape, Shift+A/B seam picker & slash-morph ── */
(() => {
  window.craftDesktopUX?.disable?.();
  if (!window.__craftCMCore.features.init('desktop-ux', 'Desktop insertion UX', 'Hover-edge drag handle, Shift+A/B seam picker, slash-morph')) return;
  const {mountTpl, qs, on, SOLVEIT} = window.__craftCMCore;
  if (!qs(SOLVEIT.sel.dialog)) return;
  const ac = new AbortController(), sig = {signal: ac.signal}, cap = {capture: true, signal: ac.signal};
  const isMobile = () => !!window.craftMobile?.active?.();
  const dialog = () => qs(SOLVEIT.sel.dialog);
  const CELL_TYPES = window.__craftCfg?.cell_types || ['Code', 'Note', 'Prompt', 'Raw'];
  const KEY_TO_TYPE = window.__craftCfg?.key_to_type || {c: 'Code', n: 'Note', p: 'Prompt', r: 'Raw'};
  const SLASH_RE = /^\/(c|code|n|note|p|prompt|r|raw)[ \t\n]$/i;
  const SLASH_MAP = window.__craftCfg?.slash_map || {c: 'Code', code: 'Code', n: 'Note', note: 'Note', p: 'Prompt', prompt: 'Prompt', r: 'Raw', raw: 'Raw'};
  /* Bottom hover-edge + handle on desktop (shown when hovering near bottom-center) */
  const hBelow = mountTpl('desk_edge_below');
  const syncHandleKinds = () => {
    const t = window.craftInsertPreview?.type?.() || 'Code';
    const k = hBelow.querySelector('.craft-desk-edge-kind');
    if (k) k.innerHTML = window.__craftTpl?.[`desk_icon_${t}`] || window.__craftTpl?.desk_icon_Code || '';
    hBelow.title = `Click to insert ${t} below · Drag to choose type`;
  };

  /* Shared 4-option fan picker for both drag-to-insert and Shift+A/B */
  const picker = mountTpl('desk_picker');
  document.body.append(picker);

  let hostCell = null, pState = null; // {edge, cell, pointerId, startX, startY, dragged, hotType, keyMode}
  const detachHandles = () => {
    hostCell?.classList.remove('craft-desk-host');
    hBelow.remove();
    hostCell = null;
  };
  const attachHandles = cell => {
    if (isMobile() || !cell?.isConnected) { detachHandles(); return; }
    if (cell === hostCell) { syncHandleKinds(); return; }
    detachHandles();
    hostCell = cell;
    cell.classList.add('craft-desk-host');
    syncHandleKinds();
    cell.append(hBelow);
  };
  const setHot = type => {
    if (pState) pState.hotType = type;
    for (const b of picker.querySelectorAll('button[data-type]')) {
      b.toggleAttribute('data-hot', b.dataset.type === type);
    }
  };
  const closePicker = () => {
    hBelow.removeAttribute('data-active');
    if (picker.matches(':popover-open')) picker.hidePopover();
    setHot(null);
    pState = null;
  };
  const positionPicker = (anchorRect, edge, hint) => {
    picker.querySelector('.craft-desk-picker-head').textContent = hint || (edge === 'above' ? 'Insert above' : 'Insert below');
    if (!picker.matches(':popover-open')) picker.showPopover();
    const pw = picker.offsetWidth || 270, ph = picker.offsetHeight || 68;
    const cx = anchorRect.left + anchorRect.width / 2;
    const left = Math.max(8, Math.min(cx - pw / 2, innerWidth - pw - 8));
    let top;
    if (edge === 'above') {
      const aboveTop = anchorRect.top - ph - 10;
      top = aboveTop >= 8 ? aboveTop : Math.min(anchorRect.bottom + 10, innerHeight - ph - 8);
    } else {
      const belowTop = anchorRect.bottom + 10;
      top = (belowTop + ph <= innerHeight - 8) ? belowTop : Math.max(8, anchorRect.top - ph - 10);
    }
    picker.style.left = Math.round(left) + 'px';
    picker.style.top = Math.round(top) + 'px';
  };
  const hitOpt = (x, y) => {
    const PAD = 8;
    for (const b of picker.querySelectorAll('button[data-type]')) {
      const r = b.getBoundingClientRect();
      if (x >= r.left - PAD && x <= r.right + PAD && y >= r.top - PAD && y <= r.bottom + PAD) return b.dataset.type;
    }
    return null;
  };
  async function commitEditors() {
    if (window.craftC2?.active?.()) await window.craftC2.exit();
    if (window.craftMd?.active?.()) await window.craftMd.exit();
  }
  async function doInsert(edge, cell, type) {
    if (type) window.craftInsertPreview?.choose?.(type);
    syncHandleKinds();
    await commitEditors();
    window.craftInsertPreview?.insert?.(edge, {type: type || window.craftInsertPreview?.type?.(), cell, force: true});
  }

  const onHandleDown = e => {
    if (isMobile() || e.button !== 0) return;
    const handle = e.currentTarget;
    const cell = handle.closest(SOLVEIT.sel.cellWithId);
    if (!cell) return;
    e.preventDefault(); e.stopPropagation();
    closePicker();
    const edge = handle.dataset.edge;
    handle.dataset.active = 'true';
    handle.setPointerCapture?.(e.pointerId);
    pState = {edge, cell, pointerId: e.pointerId, startX: e.clientX, startY: e.clientY, dragged: false, hotType: null, keyMode: false};
  };
  const onHandleMove = e => {
    if (!pState || e.pointerId !== pState.pointerId || pState.keyMode) return;
    const dist = Math.hypot(e.clientX - pState.startX, e.clientY - pState.startY);
    if (!pState.dragged && dist > 6) {
      pState.dragged = true;
      positionPicker(e.currentTarget.getBoundingClientRect(), pState.edge, `Drop to insert ${pState.edge} & set default`);
    }
    if (pState.dragged) setHot(hitOpt(e.clientX, e.clientY));
  };
  const onHandleUp = e => {
    if (!pState || e.pointerId !== pState.pointerId || pState.keyMode) return;
    const {edge, cell, dragged, hotType} = pState;
    closePicker();
    if (!dragged) doInsert(edge, cell, null);
    else if (hotType) doInsert(edge, cell, hotType);
  };
  const onHandleCancel = e => { if (pState && e.pointerId === pState.pointerId) closePicker(); };
  const onHandleClick = e => { e.preventDefault(); e.stopPropagation(); };
  on(hBelow, {pointerdown: onHandleDown, pointermove: onHandleMove, pointerup: onHandleUp, pointercancel: onHandleCancel, click: onHandleClick}, sig);
  const onPickerClick = e => {
    const b = e.target.closest('button[data-type]');
    if (!b || !pState) return;
    e.preventDefault(); e.stopPropagation();
    const {edge, cell} = pState;
    closePicker();
    doInsert(edge, cell, b.dataset.type);
  };
  on(picker, 'click', onPickerClick, sig);
  on(picker, 'toggle', e => { if (e.newState === 'closed' && pState) closePicker(); }, sig);

  /* Attach bottom hover-edge handle to the hovered cell on desktop (CSS :hover reveals the pill) */
  const onDialogOver = e => {
    if (isMobile() || pState) return;
    const cell = e.target instanceof Element ? e.target.closest(SOLVEIT.sel.cellWithId) : null;
    if (cell && cell !== hostCell) attachHandles(cell);
  };
  const onDialogLeave = () => { if (!pState) detachHandles(); };
  on(dialog(), 'mouseover', onDialogOver, sig);
  on(dialog(), 'mouseleave', onDialogLeave, sig);

  /* Click-margin / click-header to escape active inline editors on desktop */
  const onDocMouseDown = e => {
    if (isMobile()) return;
    if (e.button !== 0) return;
    const hasEd = !!(window.craftC2?.active?.() || window.craftMd?.active?.());
    if (!hasEd) return;
    const t = e.target;
    if (!(t instanceof Element)) return;
    if (t.closest('#solveit-file-tree-panel, #solveit-temp-editor, #solveit-launcher, .craft-action-menu, #craft-desk-picker, .craft-desk-edge, .cm-editor, .cm-tooltip, .uk-modal, .uk-dropdown, .uk-drop, nav.monster-navbar')) return;
    const inDialog = t.closest(SOLVEIT.sel.dialog);
    if (!inDialog) return;
    const inCell = t.closest(SOLVEIT.sel.cellWithId);
    const onHeaderBlank = inCell && t.closest(SOLVEIT.sel.cardHeader) && !t.closest('button, a, input, select');
    if (!inCell || onHeaderBlank) {
      commitEditors();
    }
  };
  on(document, 'mousedown', onDocMouseDown, cap);

  /* Shift+A / Shift+B inline seam quick-picker in selection mode */
  const blocked = () => window.__craftCMCore.typingBlocked('.cm-editor');
  const onKeyDown = e => {
    if (isMobile() || e.isComposing) return;
    if (pState?.keyMode) {
      if (e.key === 'Escape') { e.preventDefault(); e.stopImmediatePropagation(); closePicker(); return; }
      const t = KEY_TO_TYPE[e.key.toLowerCase()];
      if (t && !e.metaKey && !e.ctrlKey && !e.altKey) {
        e.preventDefault(); e.stopImmediatePropagation();
        const {edge, cell} = pState;
        closePicker();
        doInsert(edge, cell, t);
        return;
      }
      return;
    }
    if (e.shiftKey && !e.metaKey && !e.ctrlKey && !e.altKey && (e.key === 'A' || e.key === 'B')) {
      if (blocked()) return;
      const cell = qs(SOLVEIT.sel.primaryCell);
      if (!cell) return;
      e.preventDefault(); e.stopImmediatePropagation();
      const edge = e.key === 'A' ? 'above' : 'below';
      const r = cell.getBoundingClientRect();
      const seamRect = edge === 'above'
        ? {left: r.left, width: r.width, top: r.top, bottom: r.top}
        : {left: r.left, width: r.width, top: r.bottom, bottom: r.bottom};
      pState = {edge, cell, pointerId: null, startX: 0, startY: 0, dragged: false, hotType: null, keyMode: true};
      positionPicker(seamRect, edge, `Insert ${edge} · press C / N / P / R`);
    }
  };
  on(document, 'keydown', onKeyDown, cap);

  /* Slash-morph on empty cells: typing /c, /code, /n, /note, /p, /prompt, /r, /raw + Space/Enter morphs the cell */
  let morphing = false;
  const checkSlashMorph = () => {
    if (morphing || isMobile()) return;
    let id = null, view = null;
    const c2Id = window.craftC2?.active?.();
    if (c2Id) {
      view = window.craftC2.view(c2Id);
      if (view?.hasFocus) id = '_' + String(c2Id).replace(/^_/, '');
    }
    if (!id) {
      const mdKey = window.craftMd?.active?.();
      if (mdKey && mdKey.endsWith(':in')) {
        view = window.craftMd.view(mdKey);
        if (view?.hasFocus) id = '_' + mdKey.split(':')[0].replace(/^_/, '');
      }
    }
    if (!id || !view) return;
    const text = view.state.doc.toString();
    const m = SLASH_RE.exec(text);
    if (!m) return;
    const targetName = SLASH_MAP[m[1].toLowerCase()];
    const targetType = targetName.toLowerCase();
    const el = document.getElementById(id);
    const curType = el?.querySelector(SOLVEIT.sel.cardInType)?.dataset.mtype;
    if (!el || curType === targetType) return;
    morphing = true;
    view.dispatch({changes: {from: 0, to: text.length, insert: ''}});
    (async () => {
      try {
        await commitEditors();
        window.craftInsertPreview?.choose?.(targetName);
        if (window.craftInsertPreview?.setType) {
          await window.craftInsertPreview.setType(id, targetType);
        }
        if (targetType === 'code') window.craftC2?.edit?.(id.slice(1), {anchor: 0, head: 0});
        else window.craftMd?.edit?.(id.slice(1), {toEnd: true});
      } catch (err) {
        console.error('craftDesktopUX: slash-morph failed', err);
      } finally { morphing = false; }
    })();
  };
  let slashPend = false;
  const onSlashInput = e => {
    const t = e.target;
    if (!(t instanceof Element) || !t.closest('.cm-content, .cm-editor')) return;
    if (slashPend) return;
    slashPend = true;
    queueMicrotask(() => { slashPend = false; checkSlashMorph(); });
  };
  on(document, 'input', onSlashInput, cap);

  const offMobile = window.craftMobile?.onChange?.(active => { if (active) { closePicker(); detachHandles(); } });
  window.craftDesktopUX = {
    disable() {
      offMobile?.(); closePicker(); detachHandles(); picker.remove();
      ac.abort();
      delete window.craftDesktopUX;
    },
  };
})();

## Image Drop Handoff (`window.craftImageDrop`)

Routes image files dropped onto a code, note, or prompt card into Solveit's native Monaco attachment flow.

In [ ]:
#| export_js CRAFT.js
#| eval: false
(() => {
  window.craftImageDrop?.disable();
  if (!window.__craftCMCore.features.init('image-drop', 'Image drop upload', 'Drop images onto code/note/prompt cards')) return;
  const {on, notify: coreNotify, SOLVEIT, kid, htmxPost, getCellInputText} = window.__craftCMCore;
  const target = e => e.target.closest?.(`${SOLVEIT.sel.cell} > .card-in:is([data-mtype="code"],[data-mtype="note"],[data-mtype="prompt"])`);
  const notify = message => coreNotify(message, {status: 'warning'});
  let busy = false, disabled = false;

  const dragover = e => {
    if (!target(e)) return;
    const items = [...(e.dataTransfer?.items || [])];
    if (!items.some(i => i.kind === 'file' && (!i.type || i.type.startsWith('image/'))) && (items.length || !e.dataTransfer?.types.includes('Files'))) return;
    e.preventDefault();
    e.dataTransfer.dropEffect = 'copy';
  };

  const insertIntoActiveView = (id, md) => {
    const view = window.craftC2?.view?.(id) || window.craftMd?.view?.(`${id}:in`) || window.craftMd?.view?.(id);
    if (!view) return false;
    const {from, to} = view.state.selection.main;
    const ins = `\n${md}\n`;
    view.dispatch({changes: {from, to, insert: ins}, selection: {anchor: from + ins.length}});
    view.focus();
    return true;
  };

  const attach = async (e, card, file) => {
    e.preventDefault();
    e.stopPropagation();
    if (busy) return notify('Wait for the current image upload to finish.');
    const cell = card.parentElement, id = cell.id.replace(/^_/, ''), kind = card.dataset.mtype;
    const k = kid();
    if (!k) return notify('No active kernel session for image upload.');
    busy = true;
    try {
      const ext = file.name.match(/\.([^.]+)$/)?.[1] || file.type.split('/')[1] || 'png';
      const fd = new FormData();
      fd.append('id_', id);
      fd.append('msg_type', kind);
      fd.append('kid', k);
      fd.append('file', file, file.name || `image.${ext}`);
      const resp = await fetch(SOLVEIT.api.uploadAttachment, {method: 'POST', body: fd});
      if (!resp.ok) throw new Error(`HTTP ${resp.status}`);
      const data = await resp.json();
      if (disabled) return;
      const ref = data.saved_file || (data.id ? `attachment:${data.id}` : '');
      if (!ref) throw new Error('Upload did not return an attachment reference.');
      const alt = (file.name || 'image').replace(/[\[\]]/g, '');
      const md = `![${alt}](${ref})`;
      if (!insertIntoActiveView(id, md)) {
        const cur = getCellInputText(cell);
        const next = cur ? `${cur.replace(/\n*$/, '')}\n${md}\n` : `${md}\n`;
        await htmxPost(SOLVEIT.api.setContents, {id_: '_' + id, ids: id, contents: JSON.stringify({[id]: next})});
      }
    } catch (err) {
      console.error('CRAFT image attachment:', err);
      notify('Could not attach the image.');
    } finally { busy = false; }
  };

  const drop = e => {
    const card = target(e), file = e.dataTransfer?.files[0];
    if (card && file?.type.startsWith('image/')) attach(e, card, file);
  };
  const ac = new AbortController();
  on(window, {dragover, drop}, {capture: true, signal: ac.signal});
  window.craftImageDrop = {disable() {
    disabled = true;
    ac.abort();
    delete window.craftImageDrop;
  }};
})();

## Insertion & Fan Picker CSS

In [ ]:
#| export_css CRAFT.css
#| eval: false
html:not(.craft-mobile) #dialog-container > .editable.craft-desk-host {
  position: relative;
  overflow: visible !important; content-visibility: visible !important; contain: none !important;
}
html:not(.craft-mobile) #dialog-container > .editable.craft-desk-host:has(> .craft-desk-edge:is(:hover, :focus-visible, [data-active="true"]))::after {
  mask-image: linear-gradient(#000, #000), linear-gradient(#000, #000);
  mask-position: 0 0, center bottom;
  mask-size: 100% 100%, 48px 6px;
  mask-repeat: no-repeat;
  mask-composite: exclude;
  -webkit-mask-image: linear-gradient(#000, #000), linear-gradient(#000, #000);
  -webkit-mask-position: 0 0, center bottom;
  -webkit-mask-size: 100% 100%, 48px 6px;
  -webkit-mask-repeat: no-repeat;
  -webkit-mask-composite: xor;
}
.craft-desk-edge { display: none; }
html:not(.craft-mobile) .craft-desk-edge {
  display: flex; align-items: center; justify-content: center;
  position: absolute; left: 50%; z-index: 110;
  width: 160px; height: 18px; padding: 0; border: 0; background: transparent;
  transform: translateX(-50%);
  pointer-events: auto;
  user-select: none; cursor: grab;
}
html:not(.craft-mobile) .craft-desk-edge[data-edge="below"] { bottom: -9px; }
html:not(.craft-mobile) .craft-desk-edge-pill {
  display: inline-flex; align-items: center; gap: 5px;
  height: 20px; padding: 0 9px; border-radius: 999px;
  background: var(--craft-surface); border: 1px solid var(--craft-accent-border); color: var(--craft-accent);
  box-shadow: 0 2px 8px #0008;
  opacity: 0;
  transition: opacity .12s ease, transform .1s ease, background .1s ease, border-color .1s ease, color .1s ease;
}
html:not(.craft-mobile) .craft-desk-edge:is(:hover, :focus-visible, [data-active="true"]) .craft-desk-edge-pill { opacity: 1; }
html:not(.craft-mobile) .craft-desk-edge-kind { display: inline-flex; align-items: center; opacity: .75; }
html:not(.craft-mobile) .craft-desk-edge:hover .craft-desk-edge-pill,
html:not(.craft-mobile) .craft-desk-edge[data-active="true"] .craft-desk-edge-pill {
  background: var(--craft-border-soft); border-color: var(--craft-accent); color: #c0caf5; transform: scale(1.06);
}

/* Shared desktop (#craft-desk-picker) & mobile (#craft-edge-picker) blurred type-picker shell */
.craft-type-picker {
  position: fixed; z-index: 1250; margin: 0;
  padding: 6px 8px 8px;
  background: #16161ef2; backdrop-filter: blur(12px); -webkit-backdrop-filter: blur(12px);
  border: 1px solid #7aa2f755; border-radius: 10px;
  box-shadow: 0 12px 32px #000c; user-select: none; -webkit-user-select: none;
}
.craft-type-picker:popover-open { display: block; }
.craft-type-picker-head {
  font: 500 10px/1 var(--craft-prose-family); color: var(--craft-muted);
  text-align: center; margin-bottom: 6px; letter-spacing: .03em; text-transform: uppercase;
}
.craft-type-picker-opts { display: flex; gap: 5px; }
.craft-type-picker button {
  gap: 5px; height: auto; padding: 6px 9px; border-radius: 7px;
  font: 500 11.5px/1 var(--craft-prose-family); -webkit-tap-highlight-color: transparent;
  transition: transform .08s ease, background .08s ease, border-color .08s ease, color .08s ease;
}
.craft-type-picker button svg { color: var(--craft-accent); }
#craft-desk-picker button kbd {
  font: 600 9.5px/1 var(--craft-code-family); color: var(--craft-muted); opacity: .75;
  padding: 1px 4px; border-radius: 3px; background: var(--craft-recessed);
}
.craft-type-picker button:is(:hover, [data-hot], :active) {
  background: #7aa2f733; border-color: var(--craft-accent); color: var(--craft-fg-strong); transform: translateY(-1px);
}

#dialog-container.craft-insert-preview { box-sizing: border-box; margin-right: 0 !important; min-width: 0; }
#craft-insert-preview { left: auto; right: 10px; top: 36px; }
#craft-insert-preview[hidden] { display: none; }
.editable.craft-opt-hidden { display: none !important; }

/* Empty-dialog starter card (desktop & mobile; hidden once any visible cell exists) */
#dialog-container:has(> .editable:not([hidden]):not(.craft-opt-hidden)) #craft-empty-add { display: none; }
#craft-empty-add {
  display: flex; flex-direction: column; align-items: center; gap: 12px;
  margin: 48px auto; padding: 18px 20px; max-width: 380px;
  background: #16161e99; border: 1px dashed #ffffff24; border-radius: 12px;
}
#craft-empty-add .craft-empty-add-label { font-size: 12.5px; color: var(--craft-muted); text-align: center; }
#craft-empty-add .craft-empty-add-btns { display: flex; gap: 8px; flex-wrap: wrap; justify-content: center; }
#craft-empty-add button { gap: 5px; height: auto; padding: 8px 12px; font-size: 13px; }
#craft-empty-add button svg { color: var(--craft-accent); }
html.craft-mobile #craft-empty-add { margin: 32px auto; padding: 14px; max-width: 340px; }

#craft-insert-preview-menu { left: auto; right: 10px; top: 66px; margin: 0; }
#craft-insert-preview-menu small { display: block; padding: 4px 10px; color: #9b9cab; }

/* Insert picker on the right edge: replaced by cell-edge bulges, so give the width back. */
html.craft-mobile #craft-insert-preview { display: none; }
html.craft-mobile #dialog-container.craft-insert-preview { margin-right: 0 !important; padding-right: 0 !important; }
/* Mobile geometry overrides for shared #craft-edge-picker */
html.craft-mobile #craft-edge-picker { z-index: 1050; border-radius: 14px; margin: 0; }
html.craft-mobile #craft-edge-picker .craft-edge-picker-head { font-size: 10.5px; }
html.craft-mobile #craft-edge-picker .craft-edge-picker-opts { align-items: stretch; gap: 6px; }
html.craft-mobile #craft-edge-picker button { flex-direction: column; justify-content: center; gap: 3px; min-width: 60px; padding: 8px 6px; border-radius: 10px; }
html.craft-mobile #craft-edge-picker button:is([data-hot], :active) { transform: scale(1.06); }

Verify `09_cell_insertion.ipynb` exports insertion modules, config, and CSS.

In [ ]:
from craft_dev.auth import REPO_ROOT
from craft_dev.build import extract_nb_assets

assets = extract_nb_assets(REPO_ROOT/'nbs/craft/09_cell_insertion.ipynb')
js = '\n'.join(code for _, code in assets['CRAFT.js'])
for sym in ('window.craftInsertPreview', 'window.craftDesktopUX', 'window.craftImageDrop', '__craftCfg'):
    assert sym in js, f"Missing {sym}"
css = '\n'.join(code for _, code in assets['CRAFT.css'])
assert '#craft-desk-picker' in css and '#craft-insert-preview' in css

## Live `fastcdp` Verification

In [ ]:
#| live
import json
from fastcdp import Rungs
from craft_dev.cdp import open_solveit

async with open_solveit() as (page, _, page_errors):
    rungs = Rungs(page)

    async with rungs('insert preview position and hotkey'):
        assert await page.eval(r"""(() => {
          const t = document.getElementById('craft-insert-preview'), sc = document.getElementById('dialog-container');
          return !!(t && sc && t.getBoundingClientRect().right <= sc.getBoundingClientRect().right);
        })()""")
        prev_type = await page.eval("window.craftInsertPreview.type()")
        await page.press('K', mod=True, shift=True)
        await page.wait_for("window.craftInsertPreview.type() === 'Note'", timeout=3)
        await page.eval(f"window.craftInsertPreview.choose({json.dumps(prev_type)})")

    assert not page_errors, f"Uncaught page errors: {page_errors}"